# Ensemble MLP Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/instadeepai/alf/blob/main/tutorials/models/ensemble_tutorial.ipynb)

_Open in Colab works once ALF is public / on PyPI; until then, use the local setup below._

Three uncertainty-aware surrogate modes on GFP brightness prediction:

- **Seed ensemble** — N independently-seeded networks
- **MC dropout** — one network, T stochastic inference passes
- **Combined** — N networks × T passes

Each section trains one variant, predicts, and plots. Section 6 compares all three.

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment or on Google Colab.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.
- To run on a **GPU**, uncomment the GPU line in the install cell.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
%pip install "alf_core @ git+https://github.com/instadeepai/alf.git#subdirectory=core" "git+https://github.com/instadeepai/alf.git#subdirectory=tools" matplotlib pandas
# GPU (optional): run this AFTER the line above to switch PyTorch to a CUDA build.
# %pip install torch --index-url https://download.pytorch.org/whl/cu128
# Once ALF is on PyPI this simplifies to e.g. `%pip install alf_tools` (no git URL).

Bring in the standard scientific stack plus the ALF building blocks used below: the `GFP` dataset, the `MLPModel` surrogate, and the `EnsembleWrapper` that composes several models into one. The `RDLogger` block simply silences RDKit's noisy deprecation warnings, which fire when `alf_tools.datasets` imports it, so the notebook output stays readable.

In [ ]:
import logging
import time

# Importing `alf_tools.datasets` pulls in RDKit, whose legacy fingerprint API emits a flood of
# "please use MorganGenerator" deprecation notices on its own C++ logger. They come from RDKit
# itself (not this notebook), so silence that logger before the import keeps the output readable.
try:
    from rdkit import RDLogger

    RDLogger.DisableLog("rdApp.*")
except ImportError:
    pass

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from alf_core import BaseDatasetConfig, Candidate, LabelledCandidates, Modality, ProblemType
from alf_tools.datasets.gfp import GFP
from alf_tools.models import (
    EnsembleWrapper,
    EnsembleWrapperConfig,
    MLPModel,
    MLPModelConfig,
    MLPTrainConfig,
)
from scipy.stats import spearmanr

logging.basicConfig(level=logging.INFO, format="%(message)s")

## 1. Run Configuration

In [ ]:
DEVICE = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
FAST_MODE = DEVICE == "cpu"
N_MEMBERS = 10
N_EPOCHS = 25 if FAST_MODE else 50
N_MC_PASSES = 10 if FAST_MODE else 20
BASE_SEED = 42
DROPOUT_P = 0.1
HIDDEN_DIMS = [128, 64]

print(f"Device : {DEVICE}")
print(f"Mode   : {'FAST (CPU)' if FAST_MODE else 'FULL (GPU)'}")
print(f"N_MEMBERS={N_MEMBERS}  N_EPOCHS={N_EPOCHS}  N_MC_PASSES={N_MC_PASSES}")

## 2. GFP Dataset

GFP (green fluorescent protein) is a standard benchmark for sequence-fitness modelling. Each of the 1 000 variants is a 714-nt nucleotide sequence; the label is median brightness. Sequences cluster around wild-type, making calibration informative: a well-calibrated model should be more uncertain on distant variants.

In [ ]:
gfp = GFP(
    BaseDatasetConfig(
        name="gfp",
        modality=Modality.SEQUENCE,
        seed=BASE_SEED,
        train_ratio=0.8,
        validation_frac=0.1,
        test_ratio=0.1,
        problem_type=ProblemType.REGRESSION,
    )
)
labelled = gfp.load_dataset()

rng = np.random.default_rng(BASE_SEED)
indices = rng.permutation(len(labelled))
n_train = int(0.8 * len(labelled))
train_idx, val_idx = indices[:n_train], indices[n_train:]
train_raw_cands, train_labels = labelled[train_idx]
val_raw_cands, val_labels = labelled[val_idx]

print(f"Sequences : {len(labelled)}")
print(f"Train : {len(train_labels)} | Val : {len(val_labels)}")
print(
    f"Brightness range : [{labelled.labels.min():.3f}, {labelled.labels.max():.3f}]  mean={labelled.labels.mean():.3f}"
)

`MLPModel` only accepts `TABULAR` or `EMBEDDING` inputs, but the GFP candidates are raw nucleotide strings. We one-hot encode each 714-nt sequence into a flat `714 × 4` vector and rewrap it as a `TABULAR` `Candidate`, giving every variant a fixed-length numeric representation the MLP can consume.

In [ ]:
# MLPModel only accepts TABULAR/EMBEDDING modality; one-hot-encode the nucleotide sequences.
NUCLEOTIDES = list("ACGT")
SEQ_LEN = len(labelled.candidates[0].data)
INPUT_DIM = SEQ_LEN * 4


def nucleotide_one_hot(seq: str) -> np.ndarray:
    arr = np.zeros(len(seq) * 4, dtype=np.float32)
    for i, nuc in enumerate(seq):
        arr[i * 4 + NUCLEOTIDES.index(nuc)] = 1.0
    return arr


def to_tabular(raw_cands: list) -> list:
    return [
        Candidate(data=nucleotide_one_hot(c.data), modality=Modality.TABULAR) for c in raw_cands
    ]


train_cands = to_tabular(train_raw_cands)
val_cands = to_tabular(val_raw_cands)
train_data = LabelledCandidates(candidates=train_cands, labels=train_labels)
val_data = LabelledCandidates(candidates=val_cands, labels=val_labels)

print(f"Seq len: {SEQ_LEN} | Input dim: {INPUT_DIM}")

Before training, we define the shared scaffolding reused by all three sections: fixed axis limits, the eight validation indices sampled for the violin panels, and three helpers. `section_metrics` summarises predictive accuracy (Spearman, MSE) and uncertainty spread, `calibration_curve` measures whether predicted intervals contain the truth at their stated rate, and `plot_section` draws the common three-panel diagnostic figure.

In [ ]:
# Shared constants and helpers used across all three ensemble sections.
LABEL_LIM = (float(labelled.labels.min()) - 0.05, float(labelled.labels.max()) + 0.05)
VIOLIN_IDX = np.linspace(0, len(val_cands) - 1, 8, dtype=int)
metrics_table: list[dict] = []

`section_metrics` summarises a set of predictions: Spearman rank correlation and MSE (accuracy), plus the mean and spread of the predicted standard deviations (how much uncertainty the ensemble expresses).

In [ ]:
def section_metrics(preds, true_labels, elapsed: float) -> dict:
    rho, _ = spearmanr(preds.means, true_labels)
    mse = float(np.mean((preds.means - true_labels) ** 2))
    std = np.sqrt(preds.variances)
    return {
        "Spearman rho": round(float(rho), 4),
        "MSE": round(mse, 4),
        "Mean std": round(float(std.mean()), 4),
        "Std of std": round(float(std.std()), 4),
        "Train time s": round(elapsed, 1),
    }

`calibration_curve` checks whether the predicted confidence intervals are trustworthy: for each nominal level α, what fraction of true labels actually fall inside the ensemble's central α interval? A well-calibrated model tracks the diagonal.

In [ ]:
def calibration_curve(empirical_dist, true_labels, n_bins: int = 10):
    alphas, observed = np.linspace(0.1, 0.9, n_bins), []
    for alpha in alphas:
        lo = np.quantile(empirical_dist, (1 - alpha) / 2, axis=1)
        hi = np.quantile(empirical_dist, (1 + alpha) / 2, axis=1)
        observed.append(float(np.mean((true_labels >= lo) & (true_labels <= hi))))
    return alphas, np.array(observed)

Each of the three method sections below calls `plot_section`, which draws the same three panels so
the modes can be compared at a glance:

- **Mean vs ground truth** (left) — predicted mean against true brightness, points coloured by
  predicted std. Tight clustering along the red `y = x` line means accurate means; the colour
  gradient shows whether the model is more uncertain where it is less accurate.
- **Member spread** (centre) — violin plots of the full prediction sample for eight validation
  candidates, with the true value (red dot) overlaid. A true value sitting inside the violin's
  bulk indicates the spread is capturing the right answer.
- **Uncertainty distribution** (right) — histogram of predicted stds across all candidates,
  summarising how confident (narrow) or hedged (wide) the mode is overall.

In [ ]:
def plot_section(preds, true_labels, title: str) -> None:
    std = np.sqrt(preds.variances)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    sc = axes[0].scatter(preds.means, true_labels, c=std, cmap="viridis", alpha=0.6, s=15)
    plt.colorbar(sc, ax=axes[0], label="std")
    axes[0].plot(LABEL_LIM, LABEL_LIM, "r--", lw=1)
    axes[0].set(
        xlim=LABEL_LIM,
        ylim=LABEL_LIM,
        xlabel="Predicted mean",
        ylabel="True brightness",
        title="Mean vs ground truth",
    )

    data_v = [preds.empirical_dist[i] for i in VIOLIN_IDX]
    axes[1].violinplot(data_v, positions=range(8), showmedians=True)
    axes[1].scatter(range(8), true_labels[VIOLIN_IDX], color="red", zorder=3, s=30, label="True")
    axes[1].set(xlabel="Val index", ylabel="Prediction", title="Member spread (8 samples)")
    axes[1].legend(fontsize=8)

    axes[2].hist(std, bins=30, edgecolor="white", color="steelblue")
    axes[2].set(xlabel="Predicted std", ylabel="Count", title="Uncertainty distribution")

    plt.tight_layout()
    plt.show()

## 3. Seed Ensemble

N networks are trained from different random initialisations. Diversity comes entirely from weight-space randomness — no stochasticity at inference. `EnsembleWrapperConfig(base_seed, n_members)` derives seeds as `[base_seed, base_seed+1, …]`.

In [ ]:
t0 = time.time()


def seed_factory(seed: int) -> MLPModel:
    return MLPModel(
        model_config=MLPModelConfig(hidden_dims=HIDDEN_DIMS, n_mc_passes=0, model_seed=seed),
        train_config=MLPTrainConfig(num_epochs=N_EPOCHS),
        device=DEVICE,
    )


seed_ensemble = EnsembleWrapper(
    model_factory=seed_factory,
    config=EnsembleWrapperConfig(base_seed=BASE_SEED, n_members=N_MEMBERS),
)
seed_ensemble.train(train_data, val_data)
seed_time = time.time() - t0
print(f"Training time: {seed_time:.1f}s")

With the ensemble trained, we predict on the validation set and run the standard three-panel diagnostic, then record this mode's metrics in the comparison table. `predict` aggregates every member's output into a single `empirical_dist`, from which the means and variances are derived.

In [ ]:
seed_preds = seed_ensemble.predict(val_cands)
plot_section(seed_preds, val_labels, "Seed Ensemble")
seed_m = section_metrics(seed_preds, val_labels, seed_time)
metrics_table.append({"Mode": "Seed ensemble", **seed_m})
print(pd.Series(seed_m).to_string())

## 4. MC Dropout Ensemble

A single network is sampled T times at inference with dropout active. Training cost is 1× — the same as a plain MLP. `dropout_seed` is unset here, so the default `model_seed` is used: predictions are reproducible across calls.

In [ ]:
t0 = time.time()


def dropout_factory(seed: int) -> MLPModel:
    return MLPModel(
        model_config=MLPModelConfig(
            hidden_dims=HIDDEN_DIMS,
            dropout=DROPOUT_P,
            n_mc_passes=N_MC_PASSES,
            model_seed=seed,
        ),
        train_config=MLPTrainConfig(num_epochs=N_EPOCHS),
        device=DEVICE,
    )


dropout_ensemble = EnsembleWrapper(
    model_factory=dropout_factory,
    config=EnsembleWrapperConfig(base_seed=BASE_SEED, n_members=1),
)
dropout_ensemble.train(train_data, val_data)
dropout_time = time.time() - t0
print(f"Training time: {dropout_time:.1f}s")

We predict with the single dropout model and draw the same three-panel figure. Here the uncertainty comes purely from the `N_MC_PASSES` stochastic forward passes rather than from multiple trained networks.

In [ ]:
dropout_preds = dropout_ensemble.predict(val_cands)
plot_section(dropout_preds, val_labels, "MC Dropout Ensemble")

MC dropout is stochastic in principle, so it is worth confirming the predictions are actually reproducible. We call `predict` twice and scatter one call's means against the other; any drift between calls would pull points off the `y = x` line.

In [ ]:
# Reproducibility check: two calls with the same model_seed return identical predictions.
preds_a = dropout_ensemble.predict(val_cands)
preds_b = dropout_ensemble.predict(val_cands)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(preds_a.means, preds_b.means, alpha=0.6, s=15, color="steelblue")
ax.plot(LABEL_LIM, LABEL_LIM, "r--", lw=1, label="y = x (perfect reproducibility)")
ax.set(
    xlim=LABEL_LIM,
    ylim=LABEL_LIM,
    xlabel="predict() call 1",
    ylabel="predict() call 2",
    title="MC dropout: reproducibility check",
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Every point lies exactly on `y = x`: two separate `predict()` calls return identical means. MC
dropout is stochastic in principle, but because `dropout_seed` is unset the model reuses its fixed
`model_seed` to seed the dropout masks, so repeated calls are deterministic. This matters for
reproducible acquisition: a candidate's score will not drift between evaluations. Set a distinct
`dropout_seed` per call only if you deliberately want fresh stochastic samples each time.

In [ ]:
dropout_m = section_metrics(dropout_preds, val_labels, dropout_time)
metrics_table.append({"Mode": "MC Dropout", **dropout_m})
print(pd.Series(dropout_m).to_string())

## 5. Combined Ensemble (Seed + Dropout)

Each of N independently-seeded networks performs T MC dropout passes at inference. Total samples per candidate = N × T. The `empirical_dist` captures both between-member (seed) and within-member (dropout) variance.

In [ ]:
t0 = time.time()


# Same per-member config as MC Dropout; N_MEMBERS (vs 1) drives the combined diversity.
def combined_factory(seed: int) -> MLPModel:
    return MLPModel(
        model_config=MLPModelConfig(
            hidden_dims=HIDDEN_DIMS,
            dropout=DROPOUT_P,
            n_mc_passes=N_MC_PASSES,
            model_seed=seed,
        ),
        train_config=MLPTrainConfig(num_epochs=N_EPOCHS),
        device=DEVICE,
    )


combined_ensemble = EnsembleWrapper(
    model_factory=combined_factory,
    config=EnsembleWrapperConfig(base_seed=BASE_SEED, n_members=N_MEMBERS),
)
combined_ensemble.train(train_data, val_data)
combined_time = time.time() - t0
print(f"Training time: {combined_time:.1f}s")
print(f"Total samples per candidate: {N_MEMBERS} × {N_MC_PASSES} = {N_MEMBERS * N_MC_PASSES}")

We predict with the combined ensemble and draw the three-panel diagnostic. Each candidate now carries `N_MEMBERS × N_MC_PASSES` samples, so the `empirical_dist` blends weight-space (seed) and inference-time (dropout) variance.

In [ ]:
combined_preds = combined_ensemble.predict(val_cands)
plot_section(combined_preds, val_labels, "Combined Ensemble (Seed + Dropout)")

The plot below decomposes where the combined ensemble's uncertainty comes from. For a single
validation candidate, it draws one density curve per member from that member's `N_MC_PASSES`
dropout samples. Two effects are visible at once: the **spread within** each curve is dropout
(inference-time) variance, while the **spacing between** curves is seed (weight-space) variance.
When the curves sit far apart, disagreement between independently-trained networks dominates;
when each curve is broad, dropout noise dominates. The combined `empirical_dist` stacks all
`N_MEMBERS × N_MC_PASSES` samples, so it captures both sources together.

In [ ]:
# Member diversity breakdown: per-member KDE for one candidate.
# Between-member spread = seed diversity; within-member spread = dropout diversity.
cand_idx = VIOLIN_IDX[4]
n_mc = N_MC_PASSES

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(N_MEMBERS):
    samples = combined_preds.empirical_dist[cand_idx, i * n_mc : (i + 1) * n_mc]
    xs = np.linspace(samples.min() - 0.05, samples.max() + 0.05, 200)
    bw = max(1.06 * samples.std() * n_mc**-0.2, 1e-4)
    kde = np.mean(
        np.exp(-0.5 * ((xs[:, None] - samples[None, :]) / bw) ** 2) / (bw * np.sqrt(2 * np.pi)),
        axis=1,
    )
    ax.plot(xs, kde, alpha=0.7, label=f"seed {BASE_SEED + i}")

ax.set(
    xlabel="Prediction",
    ylabel="Density",
    title=f"Member diversity (candidate {cand_idx}): within=dropout, between=seed",
)
ax.legend(ncol=2, fontsize=8, title="Ensemble seeds")
plt.tight_layout()
plt.show()

Reading the curves confirms the two-source story: where the per-seed curves sit far apart, weight-space disagreement between independently-trained networks dominates this candidate's uncertainty; where each curve is broad, dropout noise dominates. Because the combined `empirical_dist` stacks all `N_MEMBERS × N_MC_PASSES` samples, it captures both effects at once — which is why it tends to give the widest, best-calibrated spread of the three modes.

Finally we record the combined mode's metrics so all three appear side by side in the comparison table that follows.

In [ ]:
combined_m = section_metrics(combined_preds, val_labels, combined_time)
metrics_table.append({"Mode": "Combined", **combined_m})
print(pd.Series(combined_m).to_string())

## 6. Comparison

The table below summarises predictive performance and uncertainty statistics for each mode. The 3×3 grid then shows scatter plots (how well means track ground truth), overlaid uncertainty distributions (how wide and spread the predicted stds are), and calibration curves (whether predicted confidence intervals actually contain the true labels at the stated rate).

In [ ]:
df = pd.DataFrame(metrics_table).set_index("Mode")


def highlight_best(col):
    styles = [""] * len(col)
    if col.name == "Spearman rho":
        best = col.idxmax()
    elif col.name in ("MSE", "Train time s"):
        best = col.idxmin()
    else:
        return styles
    styles[df.index.get_loc(best)] = "font-weight: bold; background-color: #d4f1d4"
    return styles


df.iloc[:15].style.apply(highlight_best)

The grid below lays the three modes out for direct comparison: a scatter row (means vs ground truth), an overlaid row of predicted-std densities, and a calibration row testing whether stated confidence intervals are honoured.

In [ ]:
modes = ["Seed ensemble", "MC Dropout", "Combined"]
all_preds = [seed_preds, dropout_preds, combined_preds]
all_stds = [np.sqrt(p.variances) for p in all_preds]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

fig = plt.figure(figsize=(15, 13))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.3)

# Row 0: scatter plots
for col, (mode, preds, std) in enumerate(zip(modes, all_preds, all_stds)):
    ax = fig.add_subplot(gs[0, col])
    sc = ax.scatter(preds.means, val_labels, c=std, cmap="viridis", alpha=0.5, s=10)
    ax.plot(LABEL_LIM, LABEL_LIM, "r--", lw=1)
    ax.set(xlim=LABEL_LIM, ylim=LABEL_LIM, title=mode, xlabel="Predicted mean")
    if col == 0:
        ax.set_ylabel("True brightness")

# Row 1: overlaid std KDEs on a single axes
ax_kde = fig.add_subplot(gs[1, :])
max_std = max(s.max() for s in all_stds)
xs = np.linspace(0, max_std * 1.1, 300)
for mode, std, color in zip(modes, all_stds, colors):
    bw = max(1.06 * std.std() * len(std) ** -0.2, 1e-6)
    kde = np.mean(
        np.exp(-0.5 * ((xs[:, None] - std[None, :]) / bw) ** 2) / (bw * np.sqrt(2 * np.pi)),
        axis=1,
    )
    ax_kde.plot(xs, kde, color=color, label=mode, lw=2)
    ax_kde.fill_between(xs, kde, alpha=0.15, color=color)
ax_kde.set(xlabel="Predicted std", ylabel="Density", title="Uncertainty distributions (overlaid)")
ax_kde.legend()

# Row 2: calibration curves
for col, (mode, preds, color) in enumerate(zip(modes, all_preds, colors)):
    ax = fig.add_subplot(gs[2, col])
    exp_conf, obs_conf = calibration_curve(preds.empirical_dist, val_labels)
    ax.plot(exp_conf, obs_conf, "o-", color=color, label=mode, lw=2)
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Expected conf.", title=mode)
    if col == 0:
        ax.set_ylabel("Observed conf.")
    ax.legend(fontsize=8)

fig.suptitle("Ensemble Mode Comparison", fontsize=14, fontweight="bold")
plt.show()

Reading the grid column by column ties the three modes together. The **scatter row** shows the
means are broadly similar across modes — point predictions do not change much, since all three
use the same architecture and training data. The differences live in the **uncertainty row**:
seed and combined ensembles typically produce a wider, more spread distribution of stds than MC
dropout alone, because weight-space diversity adds genuine disagreement that dropout masks cannot.
The **calibration row** is the practical test: a curve hugging the diagonal means a stated X%
confidence interval really does contain the truth X% of the time. MC dropout often sits below the
diagonal (over-confident), while seed and combined ensembles usually track it more closely. The
takeaway: if you only need a point estimate, the cheapest mode is fine; if downstream acquisition
relies on trustworthy uncertainty, the extra cost of a seed or combined ensemble buys better
calibration.

## Summary

**Seed ensembles** offer the richest weight-space diversity at N× training cost — the go-to when compute budget allows. **MC dropout** is a cheap proxy: one training run, uncertainty from random dropout masks; calibration is typically weaker. **Combined** ensembles maximise sample richness (N×T per candidate), useful for acquisition functions like Thompson sampling that draw directly from `empirical_dist`. See [`EnsembleWrapperConfig`](../../tools/alf_tools/models/ensemble.py) for `member_seeds` and other options.